# Laboratorio 01 — La tabla que recuerda

**Duración: 45 minutos.**

En el laboratorio anterior creaste una tabla y le escribiste datos. Terminamos con
una pregunta:

> Si Araucaria Ferretería pasa de PEQUENA a MEDIANA y corriges la fila,
> **¿se puede saber qué decía antes?**

Hoy la respondemos. Y no es una curiosidad técnica: es exactamente lo que la DGT no
puede responder hoy. Cuando corrigen el dato de un contribuyente, el valor anterior
desaparece. Si Contraloría pregunta qué decía ese registro en junio, no hay con qué
responder.

---

### La libreta, otra vez

Piensa en la libreta del almacén de la esquina.

- Cuando el almacenero anota algo nuevo **no borra la página anterior**: escribe una
  página nueva, fechada, y esa pasa a ser la que manda. Cada una de esas páginas es
  lo que en Iceberg se llama **snapshot**.
- **Mirar una página anterior** es ir hacia atrás en la libreta a leer. La libreta no
  cambia: sigue mandando la última. Eso es el **viaje en el tiempo**.
- **Volver a una página anterior** es otra cosa: es declarar que la página de ayer
  vuelve a ser la vigente. Eso **sí** cambia cuál manda. Eso es la **reversión**.

Mirar atrás y volver atrás no son lo mismo. Esa diferencia es todo el laboratorio.

---

### Cómo se usa este notebook

Igual que el anterior: las celdas vienen con `%%sql` en la primera línea y tú
escribes la sentencia debajo, **una por celda**, y se ejecutan con **Shift + Enter**.

La primera celda demora unos segundos: es Spark despertando, e imprime dos o tres
líneas de aviso que **no son errores**.

**Una advertencia propia de hoy.** En tres celdas vas a tener que escribir un dato
que sale de **tu propia pantalla**: el identificador de uno de tus snapshots. No
está en el notebook, y el de la solución no te sirve: cada tabla genera
los suyos. Cuando llegue el momento se dice claramente de dónde copiarlo.


## Paso 0 — Preparar la tabla

Este laboratorio **no depende de ninguno anterior**: creamos aquí todo lo que
necesitamos.

Primero, ponte en tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Por si ya habías corrido este laboratorio antes, partimos de cero. `IF EXISTS`
significa *"si existe, bórrala; si no, no pasa nada"*, así que es seguro ejecutarlo
aunque sea tu primera vez.

In [ ]:
%%sql
-- Celda 0.2



Ahora la tabla del laboratorio. Tres columnas, y otra vez `USING iceberg`, que es
lo que hace que la tabla tenga memoria.

In [ ]:
%%sql
-- Celda 0.3



Cuatro contribuyentes del padrón de la DGT, uno por segmento.

In [ ]:
%%sql
-- Celda 0.4



Comprobemos qué quedó cargado.

El `ORDER BY rut` no es decoración: sin él, el orden de las filas no está garantizado
y cada vez que lo corras podrían salir en distinto orden. Con él, sale siempre igual.

In [ ]:
%%sql
-- Celda 0.5



## Paso 1 — Corregir un dato

Araucaria Ferretería creció y la DGT la reclasifica: pasa de **PEQUENA** a
**MEDIANA**. Es el trámite más común del mundo en una administración tributaria.

Corregimos su fila.

In [ ]:
%%sql
-- Celda 1.1



Y miramos cómo quedó.

In [ ]:
%%sql
-- Celda 1.2



## Paso 2 — Descubrir que hay historia

Araucaria ahora dice MEDIANA. **PEQUENA desapareció de la vista.** Hasta aquí,
esto es lo que haría cualquier base de datos, y es exactamente el problema de la DGT.

Pero la tabla es Iceberg, así que cada escritura dejó su página en la libreta. Vamos
a pedir la lista de páginas.

Fíjate en el nombre de la tabla: **`mi_espacio.contribuyentes_lab01.snapshots`**, con las tres partes.
`snapshots` no es una tabla tuya, es una *vista del sistema* que cuelga de la tuya, y
para llegar a ella hay que nombrar la tabla completa —tu espacio, tu tabla, y luego
`snapshots`— aunque ya estés posicionado en tu espacio.

In [ ]:
%%sql
-- Celda 2.1



## Paso 3 — Mirar el pasado

Dos páginas:

- La primera, `append`, es cuando cargaste los cuatro contribuyentes.
- La segunda, `overwrite`, es la corrección de Araucaria.

**Copia ahora el `snapshot_id` de la PRIMERA fila** —la `append`, la de arriba— y
ténlo a mano: lo vas a usar tres veces. Es un número largo, y **es tuyo**: si
vuelves a crear la tabla, sale otro.

Con ese número le pedimos a la tabla que nos muestre cómo estaba **en esa página**.
Donde dice `<TU_SNAPSHOT_ID>` escribe tu número, sin comillas.

In [ ]:
%%sql
-- Celda 3.1



**Ahí está PEQUENA otra vez.** Eso es lo que la DGT no puede hacer hoy.

Ahora la comprobación que importa: consulta la tabla **normalmente**, sin pedir
ninguna versión.

In [ ]:
%%sql
-- Celda 3.2



Sigue diciendo MEDIANA.

**Mirar atrás no cambió nada.** Leíste una página anterior de la libreta y la libreta
quedó igual: sigue mandando la última página. Esto se puede hacer en producción, un
martes a las 10 de la mañana, sin riesgo.

Hay otra forma de preguntar, y es la que usaría de verdad un auditor: **por fecha y
hora**. Nadie llega preguntando por un número de snapshot; preguntan *"¿cómo estaba
esto el 30 de junio?"*.

Copia el `committed_at` de la **primera** fila de tus snapshots, tal como aparece en
pantalla, y escríbelo entre comillas donde dice `<TU_COMMITTED_AT>`.

In [ ]:
%%sql
-- Celda 3.3



## Paso 4 — Volver atrás de verdad

Mismo resultado, preguntando por tiempo en vez de por identificador.

Hasta aquí solo **leímos** el pasado. Ahora vamos a **volver** a él.

El caso real: un proceso nocturno corrió mal y dejó la tabla con datos corruptos. En
el mundo de la DGT hoy, eso significa restaurar desde un respaldo y perder horas. Con
Iceberg no se restaura nada: se **declara vigente una página anterior**.

Eso se hace con un procedimiento del sistema. Fíjate en la forma: el nombre de la
tabla va **entre comillas**, y el identificador **sin** comillas.

In [ ]:
%%sql
-- Celda 4.1



Devolvió dos números: de qué página venía y a cuál quedó apuntando.

Ahora consulta la tabla **normalmente**, sin pedir ninguna versión.

In [ ]:
%%sql
-- Celda 4.2



**PEQUENA volvió**, y esta vez sin pedir ninguna versión: es lo que la tabla dice
ahora. Esta sí cambió cuál página manda.

Última pregunta: ¿se perdió la corrección? Miremos el historial.

In [ ]:
%%sql
-- Celda 4.3



---

## Listo

Tres filas en el historial, y la del medio —la corrección que revertiste— marcada con
`is_current_ancestor` en `false`. Eso significa: *"esta página existió, pero ya no
está en la línea que manda"*.

**No se borró nada.** Si quisieras, podrías volver a mirarla con `VERSION AS OF`, o
incluso volver a ella. La libreta no pierde páginas.

### Lo que te llevas

| Lo que hiciste | Con qué | Qué cambió |
|---|---|---|
| Ver la lista de páginas | `.snapshots` | Nada |
| Mirar el pasado por identificador | `VERSION AS OF` | **Nada** |
| Mirar el pasado por fecha y hora | `TIMESTAMP AS OF` | **Nada** |
| Volver al pasado | `CALL ... rollback_to_snapshot` | **Cuál página manda** |

Las tres primeras son de lectura y se pueden hacer en producción sin riesgo. La última
es una operación de administrador: cambia el estado de la tabla.

Y con esto, la DGT ya puede responderle a Contraloría.

### La pregunta del laboratorio siguiente

Todo esto funcionó. Pero, **¿dónde está guardada esa memoria?** La tabla no tiene una
columna con las versiones. Cuando corregiste una fila, ¿qué se escribió realmente en
el almacenamiento?

Eso es el **Laboratorio 02 — Cómo está hecha la libreta**.
